# 02 — Synthetic FITS workflow

This notebook validates BayMorph's FITS-facing workflow using a controlled synthetic Sérsic image.

Workflow:

1. generate a synthetic Sérsic observation;
2. save and reload it as FITS;
3. build a source mask;
4. compare background estimators;
5. fit a Sérsic model with `DynestyFitter` and `ParameterSet`;
6. inspect posterior summaries and residuals.

The synthetic image has zero true background, so background estimates are used diagnostically here rather than subtracted before fitting.

## Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from baymorph import (
    DynestyFitter,
    ImageData,
    Parameter,
    ParameterSet,
    SersicModel,
    SyntheticSersic,
    corner_plot,
    detect_sources_mask,
    estimate_background_photutils,
    estimate_background_sigma,
    make_constant_noise_map,
    make_observation,
    plot_data_model_residual,
)

In [ ]:
from matplotlib.colors import LogNorm

BAYMORPH_IMAGE_CMAP = "turbo"


def _log_limits(
    *images,
    lower_percentile=0.5,
    upper_percentile=99.9,
    floor=1e-3,
):
    """Return robust positive limits for logarithmic image display."""
    positive_values = []

    for image in images:
        values = np.asarray(
            image,
            dtype=float,
        )
        values = values[
            np.isfinite(values)
            & (values > 0)
        ]

        if values.size > 0:
            positive_values.append(values)

    if len(positive_values) == 0:
        return floor, 1.0

    positive_values = np.concatenate(
        positive_values,
    )

    vmin = np.percentile(
        positive_values,
        lower_percentile,
    )
    vmax = np.percentile(
        positive_values,
        upper_percentile,
    )

    vmin = max(
        float(vmin),
        floor,
    )
    vmax = max(
        float(vmax),
        vmin * 10,
    )

    return vmin, vmax


def show_log_image(
    image,
    title,
    cmap=BAYMORPH_IMAGE_CMAP,
    colorbar_label="Flux",
):
    """Display an image using a robust logarithmic normalization."""
    vmin, vmax = _log_limits(
        image,
    )

    display_image = np.clip(
        np.asarray(
            image,
            dtype=float,
        ),
        vmin,
        vmax,
    )

    plt.figure(
        figsize=(6, 6),
    )

    plt.imshow(
        display_image,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )

    plt.colorbar(
        label=colorbar_label,
    )

    plt.title(
        f"{title} (log scale)",
    )

    plt.show()


def plot_log_data_model_residual(
    data,
    model,
    title="",
    cmap=BAYMORPH_IMAGE_CMAP,
):
    """Plot data and model on a log scale, with a linear residual panel."""
    data = np.asarray(
        data,
        dtype=float,
    )

    model = np.asarray(
        model,
        dtype=float,
    )

    residual = data - model

    vmin, vmax = _log_limits(
        data,
        model,
    )

    data_display = np.clip(
        data,
        vmin,
        vmax,
    )

    model_display = np.clip(
        model,
        vmin,
        vmax,
    )

    residual_limit = np.nanpercentile(
        np.abs(residual),
        99,
    )

    if not np.isfinite(residual_limit) or residual_limit == 0:
        residual_limit = 1.0

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(15, 4),
    )

    data_im = axes[0].imshow(
        data_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[0].set_title(
        "Data (log)"
    )
    fig.colorbar(
        data_im,
        ax=axes[0],
    )

    model_im = axes[1].imshow(
        model_display,
        origin="lower",
        cmap=cmap,
        norm=LogNorm(
            vmin=vmin,
            vmax=vmax,
        ),
    )
    axes[1].set_title(
        "Model (log)"
    )
    fig.colorbar(
        model_im,
        ax=axes[1],
    )

    residual_im = axes[2].imshow(
        residual,
        origin="lower",
        cmap="RdBu_r",
        vmin=-residual_limit,
        vmax=residual_limit,
    )
    axes[2].set_title(
        "Residual"
    )
    fig.colorbar(
        residual_im,
        ax=axes[2],
    )

    if title:
        fig.suptitle(
            title,
            y=1.02,
        )

    plt.tight_layout()
    plt.show()


## Configuration

In [ ]:
output_dir = Path("../outputs")
output_dir.mkdir(
    exist_ok=True,
)

mock_path = output_dir / "mock_sersic.fits"

true_parameters = {
    "flux": 1000.0,
    "re": 8.0,
    "n": 2.0,
    "q": 0.8,
    "theta": 0.0,
    "x0": 30.0,
    "y0": 30.0,
}

## Generate synthetic observation

In [ ]:
galaxy = SyntheticSersic(
    **true_parameters,
)

observation = make_observation(
    galaxy,
    shape=(60, 60),
    psf_fwhm=None,
    poisson_noise=False,
    read_noise_sigma=None,
    seed=42,
)

## Save and reload FITS

In [ ]:
observation.to_fits(
    mock_path,
    overwrite=True,
)

image = ImageData.from_fits(
    mock_path,
)

print("FITS path:", mock_path)
print("Shape:", image.shape)
print("Min / max:", image.data.min(), image.data.max())

In [ ]:
show_log_image(
    image.data,
    title="Reloaded synthetic FITS image",
)

## Source mask

In [ ]:
mask = detect_sources_mask(
    image.data,
)

print("Masked pixels:", mask.sum())
print("Mask fraction:", mask.mean())

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(
    mask,
    origin="lower",
)
plt.colorbar(label="Masked")
plt.title("Detected source mask")
plt.show()

## Background comparison

In [ ]:
simple_background = estimate_background_sigma(
    image.data,
)

phot_background = estimate_background_photutils(
    image.data,
    box_size=10,
)

masked_phot_background = estimate_background_photutils(
    image.data,
    box_size=10,
    mask=mask,
)

print("Simple sigma-clipped background:", simple_background.background)
print("Photutils background:", phot_background.background)
print("Masked photutils background:", masked_phot_background.background)

Because this is a controlled synthetic mock with zero true background, the fit below uses the raw reloaded image. The background estimates above are kept as diagnostics and as preparation for real FITS workflows.

## Prepare data and noise

In [ ]:
data = image.data.copy()

noise = make_constant_noise_map(
    data.shape,
    sigma=1.0,
)

## Model factory

In [ ]:
def model_factory(
    flux,
    re,
    n,
):
    return SersicModel(
        flux=flux,
        re=re,
        n=n,
        q=true_parameters["q"],
        theta=true_parameters["theta"],
        x0=true_parameters["x0"],
        y0=true_parameters["y0"],
    )

## ParameterSet

In [ ]:
parameters = ParameterSet(
    [
        Parameter(
            "flux",
            100.0,
            2000.0,
        ),
        Parameter(
            "re",
            2.0,
            20.0,
        ),
        Parameter(
            "n",
            0.5,
            8.0,
        ),
    ]
)

## Fit

In [ ]:
fitter = DynestyFitter(
    model_factory=model_factory,
    data=data,
    noise=noise,
    parameters=parameters,
)

result = fitter.run(
    nlive=150,
    sample="rwalk",
    print_progress=True,
)

## Posterior summaries

In [ ]:
for name in parameters.names:
    print("True", name + ":", true_parameters[name])
    print(result.summary(name))
    print()

## Corner plot

In [ ]:
fig, ax_grid, params = corner_plot(
    result.posterior_samples.T,
    parameter_labels=[
        "Flux",
        "Re",
        "n",
    ],
    bins=30,
    write_medians=True,
)

## Median-posterior model

In [ ]:
median_parameters = result.median_parameters()
median_parameters

In [ ]:
plot_log_data_model_residual(
    data,
    median_image,
    title="Median-parameter model",
)

## Best-likelihood model

In [ ]:
best_parameters = result.best_parameters
best_parameters

In [ ]:
plot_log_data_model_residual(
    data,
    best_image,
    title="Best-likelihood model",
)

## Recovery check

In [ ]:
for name in parameters.names:
    true_value = true_parameters[name]
    recovered_value = best_parameters[name]
    error_percent = 100 * (recovered_value - true_value) / true_value

    print(f"{name}: true={true_value:.6g}, recovered={recovered_value:.6g}, error={error_percent:.3g}%")

## Conclusion

This notebook validates the synthetic FITS pathway. The same modelling and fitting objects used in pure synthetic tests also work after saving and reloading through FITS.